Avide lecteur de *L'Enracinement* de Simone Weil, et depuis peu creusant le sujet de la *Fin du village* (Le Goff), étant en outre bretonnant et curieux de comprendre « l'âme » des populations de mon Finistère natal, je me propose : 1° Télécharger les réponses de plusieurs paroisses finisteriennes en 1902 à une enquête diocésaine relative à la pratique du breton/compréhension de la langue française au catéchisme en 1902 2° D'effectuer une analyse HTR de l'écriture cursive dans le but de construire un dataframe 3° D'essayer de voir s'il y a une corrélation entre les votes à cette époque et cette enquête.

De la même manière, disposant de la liste électorale actuelle du Finistère (accessible légalement pour tout citoyen inscrit sur cette liste) et du registre des décès, je désirerais mesurer l'endogamie propre aux communes bretonnes (fait de naître, de vivre et de mourir dans la même commune ou proche) et son effet sur le vote.

Cette tentative est assez déplaisante puisque l'utilisation de Kraken, nécessite des librairies d'une version de python précédente. De même, eScriptorium, logiciel libre de l'inria, a un emploi qui nécessite soit un compte qu'il est impossible de glaner sans accès au monde universitaire, ou de l'héberger soi-même, chose que j'ai faite.

D'ici

**1°**

In [4]:
import requests
from bs4 import BeautifulSoup
import os
import time
import re
import unicodedata
import json

# --- CONFIGURATION ---
BASE_URL = "https://bibliotheque.diocese-quimper.fr"
# L'URL spécifique que vous m'avez donnée
START_URL = "https://bibliotheque.diocese-quimper.fr/items/browse?collection=89"
OUTPUT_DIR = "images_enquete_1902"

# Headers pour simuler un vrai navigateur (évite parfois les blocages)
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

# Création du dossier de sauvegarde
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)

def nettoyer_nom_commune(texte):
    """
    Extrait le nom de la commune du titre et le nettoie.
    Ex: "Enquête sur le breton en 1902 : Taulé" -> "TAULE"
    """
    if ":" in texte:
        nom = texte.split(":")[-1].strip()
    else:
        nom = texte.strip()
    
    # Suppression accents
    nom = unicodedata.normalize('NFKD', nom).encode('ASCII', 'ignore').decode('utf-8')
    # Garde uniquement lettres et tirets
    nom = re.sub(r'[^\w\s-]', '', nom).strip().replace(' ', '-')
    return nom.upper()

def recuperer_code_insee_api(nom_commune):
    """
    Interroge l'API Geo Gouv pour trouver le code INSEE réel.
    """
    try:
        # On nettoie un peu le nom pour l'API (remplace les tirets par espaces pour la recherche)
        search_name = nom_commune.replace('-', ' ')
        url_api = f"https://geo.api.gouv.fr/communes?nom={search_name}&fields=code&boost=population&limit=1"
        
        req = requests.get(url_api)
        data = req.json()
        
        if data and len(data) > 0:
            return data[0]['code']
        else:
            return "00000" # Code par défaut si non trouvé
    except Exception:
        return "00000"

def telecharger_image(url_item):
    try:
        response = requests.get(url_item, headers=HEADERS)
        soup = BeautifulSoup(response.content, 'html.parser')

        # 1. Récupération du Titre (Commune)
        titre_h1 = soup.find('h1')
        if not titre_h1:
            return
        
        nom_brut = titre_h1.text.strip()
        nom_commune = nettoyer_nom_commune(nom_brut)
        
        # 2. Récupération du Code INSEE via API
        code_insee = recuperer_code_insee_api(nom_commune)

        # 3. Récupération du lien de l'image originale
        # On cherche la div qui contient les images
        image_div = soup.find('div', id='item-images')
        img_url = None
        
        if image_div:
            # On cherche le lien direct (souvent le premier <a> dans cette div pointe vers le fichier original)
            link_tag = image_div.find('a')
            if link_tag and 'href' in link_tag.attrs:
                img_url = link_tag['href']
        
        if not img_url:
            print(f"[PAS D'IMAGE] Ignoré : {nom_commune}")
            return

        # Correction de l'URL si elle n'est pas absolue
        if not img_url.startswith('http'):
            # Parfois l'URL est relative, parfois absolue sur Omeka
            # Si elle commence par /, on ajoute le domaine, sinon on laisse
            if img_url.startswith('/'):
                img_url = BASE_URL + img_url # Cas rare mais possible

        # 4. Téléchargement et Renommage
        extension = img_url.split('.')[-1].lower()
        if len(extension) > 4: extension = "jpg" # Sécurité si l'extension est bizarre

        nom_fichier = f"enquete_breton_1902_{nom_commune}_{code_insee}.{extension}"
        chemin_complet = os.path.join(OUTPUT_DIR, nom_fichier)

        # On vérifie si le fichier existe déjà pour gagner du temps
        if os.path.exists(chemin_complet):
            print(f"[DÉJÀ FAIT] {nom_fichier}")
            return

        print(f"Téléchargement : {nom_commune} ({code_insee})...")
        img_data = requests.get(img_url, headers=HEADERS).content
        with open(chemin_complet, 'wb') as handler:
            handler.write(img_data)
        
        print(f"   -> Sauvegardé : {nom_fichier}")

    except Exception as e:
        print(f"[ERREUR] Sur {url_item} : {e}")

def main():
    current_url = START_URL
    page_count = 1

    while current_url:
        print(f"\n--- Traitement de la page {page_count} ---")
        print(f"URL: {current_url}")
        
        response = requests.get(current_url, headers=HEADERS)
        soup = BeautifulSoup(response.content, 'html.parser')

        # Sur la page /items/browse, les éléments sont des divs avec la classe "item hentry"
        items = soup.find_all(class_="item")
        
        if not items:
            print("Aucun item trouvé sur cette page.")
            break

        for item in items:
            # On cherche le lien vers la fiche détaillée
            # Généralement dans un h2 > a, ou direct a.permalink
            link_tag = item.find('a', class_='permalink')
            if not link_tag:
                h2 = item.find('h2')
                if h2: link_tag = h2.find('a')
            
            if link_tag and 'href' in link_tag.attrs:
                full_item_url = link_tag['href']
                if not full_item_url.startswith('http'):
                    full_item_url = BASE_URL + full_item_url
                
                telecharger_image(full_item_url)
                # Petite pause pour ne pas surcharger le serveur du diocèse + l'API Gouv
                time.sleep(1) 

        # Gestion de la pagination (Page Suivante)
        # On cherche le lien "Suivant" dans la pagination
        next_page_link = None
        
        # Structure pagination Omeka classique
        pagination_div = soup.find('div', class_='pagination') or soup.find('ul', class_='pagination')
        
        if pagination_div:
            # Cherche lien avec classe 'next' ou dans un li class='pagination_next'
            next_a = pagination_div.find('a', class_='next')
            if not next_a:
                next_li = pagination_div.find('li', class_='pagination_next')
                if next_li: next_a = next_li.find('a')
            
            if next_a and 'href' in next_a.attrs:
                next_page_link = next_a['href']

        if next_page_link:
            # Construction de l'URL absolue si nécessaire
            if not next_page_link.startswith('http'):
                current_url = BASE_URL + next_page_link
            else:
                current_url = next_page_link
            page_count += 1
        else:
            print("\nPas de page suivante. Fin du scraping.")
            current_url = None

if __name__ == "__main__":
    main()


--- Traitement de la page 1 ---
URL: https://bibliotheque.diocese-quimper.fr/items/browse?collection=89
[DÉJÀ FAIT] enquete_breton_1902_EMPLOIABUSIFJPG_00000.jpg
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-TAULE-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-SAINT-THEGONNEC-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-SAINT-RENAN-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-SAINT-POL-DE-LEON-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-SIZUN-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-SCAER-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-ROSPORDEN-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAIT] enquete_breton_1902_REPONSE-DU-DOYENNE-DE-QUIMPERLE-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf
[DÉJÀ FAI

**2°**

In [ ]:
import os
from pdf2image import convert_from_path
from kraken import binarization, pageseg, rpred
from kraken.lib import models
# CORRECTION 1 : On importe la bonne fonction de sauvegarde
from kraken.serialization import serialize

# --- 1. CONFIGURATION ---
pdf_path = '/home/castro/Documents/ENSAE 2A/Python/Projet/Python_DS_Vote/images_enquete_1902/enquete_breton_1902_REPONSE-DU-DOYENNE-DE-TAULE-A-LEVEQUE-DE-QUIMPER-ET-LEON_00000.pdf'
model_path = '/home/castro/Documents/ENSAE 2A/Python/Projet/Python_DS_Vote/images_enquete_1902/lectaurep_base.mlmodel'

base_path = os.path.splitext(pdf_path)[0]
image_output_path = base_path + ".jpg"
xml_output_path = base_path + ".xml"

# --- 2. CHARGEMENT DU MODÈLE ---
print(f"Chargement du modèle : {os.path.basename(model_path)}...")
model = models.load_any(model_path)

# --- 3. CONVERSION PDF -> IMAGE ---
print(f"Conversion du PDF en image...")
try:
    pages = convert_from_path(pdf_path)
    if not pages:
        raise ValueError("Le PDF semble vide.")
    im = pages[0]
    print(f"Sauvegarde de l'image : {os.path.basename(image_output_path)}")
    im.save(image_output_path, 'JPEG')
except Exception as e:
    print(f"Erreur conversion PDF : {e}")
    exit()

# --- 4. BINARISATION ---
print("Binarisation...")
bw_im = binarization.nlbin(im)

# --- 5. SEGMENTATION ---
print("Segmentation...")
seg = pageseg.segment(bw_im, text_direction='horizontal-lr')

# --- 6. RECONNAISSANCE ---
print("Lecture du texte...")
# On récupère tous les résultats de l'OCR dans une liste
preds = list(rpred.rpred(model, bw_im, seg))

# ÉTAPE CRUCIALE : On "colle" le texte lu sur les lignes géométriques
# On fusionne l'intelligence (preds) avec la structure (seg)
for line, record in zip(seg.lines, preds):
    line.text = record.prediction
    line.cuts = record.cuts
    line.confidences = record.confidences

# --- 7. GÉNÉRATION XML (ALTO) ---
print("Génération du XML...")

# On s'assure que le nom de l'image est bien inscrit dans l'objet
seg.imagename = image_output_path

# On sérialise l'objet 'seg' (qui contient maintenant la géométrie ET le texte)
xml_content = serialize(
    seg, 
    image_size=im.size,  # On ajoute la taille de l'image pour un XML valide
    template='alto'
)

# --- 8. SAUVEGARDE ---
with open(xml_output_path, 'w', encoding='utf-8') as f:
    f.write(xml_content)

print(f"✅ SUCCÈS ! Fichier XML généré : {xml_output_path}")

: 

**3°** : on observe le résultat des élections législatives de 1902, pour déjà se donner une idée. On va très vite remarquer que le vote est largement une affaire de conscience d'appartenance communautaire. Il est largement celui d'une communauté plutot que d'individus citoyens.

Avec les données récoltées sur la pratique du catéchisme au point précédent, on va aussi dresser en pourcentage de la part d'enfants, les communes qui ont la plus grande vitalité catholique